# 광역의회 정책 이슈 확산 분석: 실행 노트북

대회 제출 규정의 순서(데이터 불러오기·전처리 → 모델 구성·학습 → 평가·추론 → 결과 확인)를 따른다.
오래 걸리는 단계는 `scripts/`를 실행하는 셀로 두었다. 이미 실행했다면 건너뛰고 결과 확인 셀만 실행해도 된다.

- 실행 환경: `conda env create -f environment.yml` → `conda activate council-diffusion`
- 제공 데이터: `data/raw/`에 zip 두 개 (README의 데이터 안내 참조)
- 출력에는 회의록 원문 인용이 나올 수 있다. 공개 저장소에는 출력을 지운 상태로 둔다.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P, R = ROOT / "data" / "processed", ROOT / "results"


def run(script, *args):
    """scripts/의 단계를 실행하고 마지막 출력 몇 줄을 보인다."""
    out = subprocess.run([sys.executable, str(ROOT / "scripts" / script), *args], cwd=ROOT,
                         capture_output=True, text=True, encoding="utf-8", errors="replace")
    print("\n".join((out.stdout + out.stderr).strip().splitlines()[-15:]))


import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. 데이터 불러오기와 전처리 (약 15분, CPU)
회의록 HTML을 평문으로 바꾸고 발언·안건을 나눈 뒤, 사본을 걸러 분석 대상을 정하고 문단으로 나눈다.

In [ ]:
RUN_PREPROCESS = False   # 처음 실행할 때만 True
if RUN_PREPROCESS:
    for s in ["02_preprocess.py", "02b_dedupe.py", "03_data_summary.py", "04_passages.py", "05_weak_labels.py", "06_label_pool.py"]:
        print("=====", s); run(s)

In [ ]:
docs = pd.read_parquet(P / "docs.parquet", columns=["doc_id", "council_short", "date", "keep"])
print("원자료 회의록", len(docs), "/ 분석 대상", int(docs.keep.sum()))
print("발언", pq.ParquetFile(P / "utterances.parquet").metadata.num_rows,
      "/ 문단", pq.ParquetFile(P / "passages.parquet").metadata.num_rows)
docs[docs.keep].groupby("council_short").size().rename("분석 대상 회의록").to_frame().T

### 1-1. 발언 분리 회귀 검사
의회별 의원 측 발언 비율, 교육청 발언 비율, 안건 표가 빈 회의록 비율, 놓친 발언자 줄 비율.

In [ ]:
run("02c_check_segmentation.py")
pd.read_csv(P / "segmentation_check.csv", index_col=0)

## 2. 모델 구성과 학습
LLM은 쓰지 않는다. 학습하는 모델은 한국어 NLI 판정기 하나다(KoELECTRA + KLUE-NLI, 약 5분, GPU).
bge-m3(이슈 묶기·쟁점 군집)와 bge-reranker-v2-m3(관련 판정)는 학습 없이 쓴다.

In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    run("07_train_nli.py")
m = json.load(open(ROOT / "model" / "koelectra-klue-nli" / "metrics.json", encoding="utf-8"))
{k: m[k] for k in m if k not in ("log", "history", "epochs")}

## 3. 추론: 핵심 이슈, 사건표, 예측, 이슈 카드 (GPU 약 30~40분)

In [ ]:
RUN_CARDS = False
if RUN_CARDS:
    run("08_issue_cards.py")

## 4. 평가와 결과 확인
### 4-1. 확산 예측 백테스트
결과 창이 겹치지 않는 기준점(2025-01-01, 2025-07-01)에서 잰 AUC, Brier 기술 점수, 등급별 실측 빈도.

In [ ]:
bt = json.load(open(R / "forecast_backtest.json", encoding="utf-8"))
display(pd.DataFrame(bt["metrics"]))
pd.DataFrame(bt["grades"]).T

### 4-2. 핵심 이슈와 대회 결과표

In [ ]:
issues = pd.read_parquet(P / "issues.parquet")
display(issues[["issue_id", "kind", "title", "theme", "aliases", "base_score"]].head(40))
table = pd.read_csv(R / "issue_table.csv")
table.head(10)

### 4-3. 이슈 카드
`results/issue_cards/index.html`을 브라우저로 열면 목차와 카드를 볼 수 있다.

In [ ]:
from IPython.display import IFrame
index = R / "issue_cards" / "index.html"
print(index)
IFrame(Path("..", "results", "issue_cards", "index.html").as_posix() if Path.cwd().name == "notebooks" else index.relative_to(ROOT).as_posix(),
       width="100%", height=600)